# Software Defect Prediction with XGBoost and K-Fold Cross-Validation

This notebook trains XGBoost classifiers on NASA software defect datasets (CM1, JM1, KC1, PC1) using Stratified K-Fold cross-validation, compares results across datasets, and generates predictions on the KC2 blind test set.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

## 2. Load Training Datasets

Load the four NASA software defect datasets. Each dataset contains software module metrics (Halstead, McCabe complexity) and a binary `defects` label (true/false).

The first two rows after the header in each file are metadata rows — we skip them.

In [ ]:
# Paths to training datasets
dataset_paths = {
    "CM1": "datasets/datasets/cm1.csv",
    "JM1": "datasets/datasets/jm1.csv",
    "KC1": "datasets/datasets/kc1.csv",
    "PC1": "datasets/datasets/pc1.csv",
}

# Common feature columns (lowercase, consistent naming)
FEATURE_COLS = [
    "loc", "v(g)", "ev(g)", "iv(g)", "n", "v", "l", "d", "i", "e", "b", "t",
    "locode", "locomment", "loblank", "loccodeandcomment",
    "uniq_op", "uniq_opnd", "total_op", "total_opnd", "branchcount",
]

datasets = {}
for name, path in dataset_paths.items():
    df = pd.read_csv(path, skiprows=[1, 2])  # skip metadata rows
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "")
    # Map defects column to binary
    df["defects"] = df["defects"].map({"true": 1, "false": 0, True: 1, False: 0}).astype(int)
    datasets[name] = df
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

datasets["CM1"].head()

## 3. Exploratory Data Analysis

Check for missing values, class distributions, and basic statistics per dataset.

In [ ]:
# Missing values and class distribution per dataset
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    class_counts = df["defects"].value_counts()
    defect_rate = class_counts.get(1, 0) / len(df) * 100
    print(f"--- {name} ---")
    print(f"  Missing values: {missing}")
    print(f"  Class distribution: No defect={class_counts.get(0, 0)}, Defect={class_counts.get(1, 0)} ({defect_rate:.1f}%)")
    print()

In [ ]:
# Class distribution bar chart
fig, axes = plt.subplots(1, len(datasets), figsize=(14, 3), sharey=True)
for ax, (name, df) in zip(axes, datasets.items()):
    df["defects"].value_counts().sort_index().plot.bar(ax=ax, color=["steelblue", "coral"])
    ax.set_title(name)
    ax.set_xlabel("Defects")
    ax.set_ylabel("Count")
    ax.set_xticklabels(["No", "Yes"], rotation=0)
plt.suptitle("Class Distribution per Dataset", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for one example dataset (CM1)
plt.figure(figsize=(12, 8))
corr = datasets["CM1"][FEATURE_COLS].corr()
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap (CM1)")
plt.tight_layout()
plt.show()

## 4. Data Preprocessing and Feature Engineering

Standardize column names across datasets, handle missing values, and prepare feature matrices (X) and targets (y).

In [ ]:
# Prepare X, y for each dataset using the common feature columns
data_splits = {}
for name, df in datasets.items():
    # Fill missing values with median
    X = df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(df[FEATURE_COLS].median())
    y = df["defects"]
    data_splits[name] = (X, y)
    print(f"{name}: X shape={X.shape}, y distribution: {dict(y.value_counts())}")

# Also create a combined dataset from all training data
X_all = pd.concat([data_splits[n][0] for n in data_splits], ignore_index=True)
y_all = pd.concat([data_splits[n][1] for n in data_splits], ignore_index=True)
print(f"\nCombined: X shape={X_all.shape}, y distribution: {dict(y_all.value_counts())}")

## 5. Define XGBoost Model and K-Fold Cross-Validation Strategy

We use `StratifiedKFold` (k=5) to preserve class distribution across folds, which is important given the imbalanced nature of defect datasets. XGBoost's `scale_pos_weight` is set to handle class imbalance.

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def make_xgb_model(scale_pos_weight=1.0):
    return XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )

print(f"K-Fold strategy: StratifiedKFold with k={N_SPLITS}")

## 6. Train XGBoost with K-Fold on Each Dataset

For each dataset, run Stratified K-Fold cross-validation, track accuracy, F1-score, and AUC per fold.

In [ ]:
results = {}

for name, (X, y) in data_splits.items():
    # Calculate class imbalance ratio for scale_pos_weight
    neg_count = (y == 0).sum()
    pos_count = (y == 1).sum()
    spw = neg_count / pos_count if pos_count > 0 else 1.0

    fold_metrics = {"accuracy": [], "f1": [], "auc": []}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = make_xgb_model(scale_pos_weight=spw)
        model.fit(X_train, y_train, verbose=False)

        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        fold_metrics["accuracy"].append(accuracy_score(y_val, y_pred))
        fold_metrics["f1"].append(f1_score(y_val, y_pred))
        fold_metrics["auc"].append(roc_auc_score(y_val, y_proba))

    results[name] = fold_metrics
    print(f"{name}:  Accuracy={np.mean(fold_metrics['accuracy']):.4f} (±{np.std(fold_metrics['accuracy']):.4f})  "
          f"F1={np.mean(fold_metrics['f1']):.4f} (±{np.std(fold_metrics['f1']):.4f})  "
          f"AUC={np.mean(fold_metrics['auc']):.4f} (±{np.std(fold_metrics['auc']):.4f})")

## 7. Evaluate Cross-Validation Results Across Datasets

Compare mean metrics across all datasets to identify which performs best, and visualize with bar charts.

In [ ]:
# Build summary table
summary_rows = []
for name, metrics in results.items():
    for metric_name in ["accuracy", "f1", "auc"]:
        summary_rows.append({
            "Dataset": name,
            "Metric": metric_name.upper(),
            "Mean": np.mean(metrics[metric_name]),
            "Std": np.std(metrics[metric_name]),
        })
summary_df = pd.DataFrame(summary_rows)
summary_pivot = summary_df.pivot(index="Dataset", columns="Metric", values="Mean")
print(summary_pivot.round(4).to_string())

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["ACCURACY", "F1", "AUC"]):
    metric_data = summary_df[summary_df["Metric"] == metric]
    ax.bar(metric_data["Dataset"], metric_data["Mean"],
           yerr=summary_df[summary_df["Metric"] == metric]["Std"].values,
           capsize=5, color="steelblue", edgecolor="black")
    ax.set_title(f"Mean {metric}")
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric)
plt.suptitle("K-Fold Cross-Validation Results per Dataset", y=1.02)
plt.tight_layout()
plt.show()

## 8. Train Final Model on All Combined Training Data

Train the final XGBoost model on all combined training data to maximize generalization for the blind test.

In [ ]:
# K-Fold CV on the combined dataset first
neg_all = (y_all == 0).sum()
pos_all = (y_all == 1).sum()
spw_all = neg_all / pos_all

combined_metrics = {"accuracy": [], "f1": [], "auc": []}
for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):
    X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_train, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]

    model = make_xgb_model(scale_pos_weight=spw_all)
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    combined_metrics["accuracy"].append(accuracy_score(y_val, y_pred))
    combined_metrics["f1"].append(f1_score(y_val, y_pred))
    combined_metrics["auc"].append(roc_auc_score(y_val, y_proba))

print("Combined dataset K-Fold CV:")
print(f"  Accuracy={np.mean(combined_metrics['accuracy']):.4f} (±{np.std(combined_metrics['accuracy']):.4f})")
print(f"  F1={np.mean(combined_metrics['f1']):.4f} (±{np.std(combined_metrics['f1']):.4f})")
print(f"  AUC={np.mean(combined_metrics['auc']):.4f} (±{np.std(combined_metrics['auc']):.4f})")

# Train final model on ALL combined data
final_model = make_xgb_model(scale_pos_weight=spw_all)
final_model.fit(X_all, y_all, verbose=False)
print("\nFinal model trained on all combined data.")

In [ ]:
# Feature importance from the final model
importance = pd.Series(final_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(8, 6))
importance.plot.barh(color="steelblue")
plt.title("XGBoost Feature Importance (Final Model)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Load and Preprocess Blind Test Dataset (KC2)

Load the KC2 blind test set, align its columns with the training features, and apply the same preprocessing.

In [ ]:
# Load blind test set
blind_df = pd.read_csv("test-blind/test-blind/kc2_test_blind.csv", skiprows=[1, 2])
blind_df.columns = blind_df.columns.str.strip().str.lower().str.replace(" ", "")

print(f"Blind test set shape: {blind_df.shape}")
print(f"Columns: {list(blind_df.columns)}")
blind_df.head()

In [ ]:
# Prepare blind test features (same columns as training)
# The blind set has an 'id' column and no 'defects' column
blind_ids = blind_df["id"] if "id" in blind_df.columns else blind_df.index

X_blind = blind_df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce")
# Fill any missing values with the median from the training data
for col in FEATURE_COLS:
    if X_blind[col].isnull().any():
        X_blind[col] = X_blind[col].fillna(X_all[col].median())

print(f"Blind test features shape: {X_blind.shape}")
print(f"Missing values: {X_blind.isnull().sum().sum()}")

## 10. Predict on Blind Test Set

Generate predictions and probabilities for the KC2 blind test set using the final model. Since this is a *blind* test, we output the predictions as a CSV file.

In [ ]:
# Predict on blind test set
blind_preds = final_model.predict(X_blind)
blind_proba = final_model.predict_proba(X_blind)[:, 1]

# Build results DataFrame
predictions_df = pd.DataFrame({
    "id": blind_ids,
    "predicted_defect": blind_preds,
    "defect_probability": blind_proba,
})
predictions_df["predicted_defect"] = predictions_df["predicted_defect"].map({1: "true", 0: "false"})

print(f"Prediction distribution:")
print(predictions_df["predicted_defect"].value_counts())
print(f"\nTotal predictions: {len(predictions_df)}")
predictions_df.head(10)

In [ ]:
# Visualize prediction probability distribution
plt.figure(figsize=(8, 4))
plt.hist(blind_proba, bins=30, color="steelblue", edgecolor="black", alpha=0.7)
plt.axvline(x=0.5, color="red", linestyle="--", label="Decision threshold (0.5)")
plt.xlabel("Predicted Defect Probability")
plt.ylabel("Count")
plt.title("Distribution of Predicted Defect Probabilities (KC2 Blind Test)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save predictions to CSV
output_path = "kc2_blind_predictions.csv"
predictions_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")